# Mortgage Risk Prediction using Machine Learning

In this project, I tried to predict loan risk using different machine learning models.

I used features like credit score, DTI, and loan-related attributes.

I started with SVC and then compared it with Logistic Regression and Random Forest.

## Importing Libraries

In [48]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np


## Loading and Exploring Data

In [49]:

df = pd.read_csv('Cleaned_LoanExport_Final.csv')

#Dropping unnecessary ID columns if present
columns_to_drop = ['LoanID', 'CustomerID']  
df.drop(columns=[col for col in columns_to_drop if col in df.columns], axis=1, inplace=True)

# Checking available columns
print(df.columns.tolist())


C:\Users\kalya\AppData\Local\Temp\ipykernel_28284\2119346254.py:1: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Cleaned_LoanExport_Final.csv')


['creditscore', 'firstpaymentdate', 'firsttimehomebuyer', 'maturitydate', 'msa', 'mip', 'units', 'ocltv', 'dti', 'origupb', 'ltv', 'originterestrate', 'ppm', 'producttype', 'propertystate', 'postalcode', 'loanseqnum', 'origloanterm', 'numborrowers', 'sellername', 'servicername', 'everdelinquent', 'monthsdelinquent', 'monthsinrepayment', 'computedmaturity', 'date_diff', 'creditscorecategory', 'dticategory', 'loanriskcategory', 'delinquencyflag', 'loancorrectionindicator', 'loanpurpose_Refinance', 'occupancy_Primary Residence', 'occupancy_Second Home', 'propertytype_CP', 'propertytype_LH', 'propertytype_MH', 'propertytype_PU', 'propertytype_SF', 'propertytype_X ', 'channel_C', 'channel_R', 'channel_T', 'loanperformancecategory_Short Delinquency', 'loanperformancecategory_Moderate Delinquency', 'loanperformancecategory_Severe Delinquency']


## Data Preprocessing

Here I am encoding target variables and preparing features.

In [50]:
#  Encoding Target Column if needed
target_column = 'loanriskcategory'

if df[target_column].dtype == 'object':
    label_enc = LabelEncoder()
    df[target_column] = label_enc.fit_transform(df[target_column])


## Feature Selection and Cleaning

Here I am selecting only relevant features for the model.

I removed unnecessary columns and kept only numerical features.

I also handled missing values by filling them with median values.

In [51]:

drop_cols = [target_column, 'firstpaymentdate', 'maturitydate', 'computedmaturity']
existing_drop_cols = [col for col in drop_cols if col in df.columns]
X = df.drop(columns=existing_drop_cols)

X = X.select_dtypes(include=['float64', 'int64'])

#  Filling missing values
X = pd.DataFrame(X).fillna(X.median())

#  Target Variable
y = df[target_column]


## Sampling Data

To speed up SVC training, I am using a smaller sample of the dataset.

This helps reduce computation time while still giving a good idea of model performance.

In [52]:
# Sampling 1000 random rows for faster SVC training
sample_size = 1000

if len(X) > sample_size:
    random_indices = np.random.choice(len(X), sample_size, replace=False)
    X = X.iloc[random_indices]
    y = y.iloc[random_indices]


## Feature Scaling

Scaling is important so that all features are on the same scale and the model performs better.

In [53]:
#  Standardizing Features
scaler = StandardScaler()
X = scaler.fit_transform(X)


## Train-Test Split

Here I split the dataset into training and testing sets.

In [54]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Support Vector Classifier (SVC)

This is my first model to understand baseline performance.

SVC works well for smaller datasets but may struggle with large data.

In [55]:
# Building and Training SVC Model
svc_model = SVC(probability=True)
svc_model.fit(X_train, y_train)


SVC(probability=True)

In [56]:
# Predictions
y_pred = svc_model.predict(X_test)

# Evaluation Metrics
print("---- Support Vector Classifier (SVC) ----")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Recall:", recall_score(y_test, y_pred, average='weighted'))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))


---- Support Vector Classifier (SVC) ----
Accuracy: 0.78
Precision: 0.7467525370804059
Recall: 0.78
F1 Score: 0.7297507677769901


### SVC Observations

The SVC model gives decent performance based on accuracy and F1 score.

However, I feel it may not capture complex relationships in the data very well.

Also, since the dataset is large, SVC might not be the most efficient model in terms of computation time.

Compared to Random Forest, its performance is slightly lower.

## Additional Models

To improve performance, I tried Logistic Regression and Random Forest.

This helps compare different approaches.

In [57]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("training additional models...")

# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("models trained")

training additional models...
models trained


In [58]:
from sklearn.metrics import accuracy_score, classification_report

print("\n--- MODEL COMPARISON ---")

# Logistic Regression
acc_lr = accuracy_score(y_test, pred_lr)
print("\nLogistic Regression Accuracy:", acc_lr)

# Random Forest
acc_rf = accuracy_score(y_test, pred_rf)
print("\nRandom Forest Accuracy:", acc_rf)

# SVC (already done but let's compare)
acc_svc = accuracy_score(y_test, y_pred)
print("\nSVC Accuracy:", acc_svc)

print("\n--- Detailed Report (Random Forest) ---")
print(classification_report(y_test, pred_rf))


--- MODEL COMPARISON ---

Logistic Regression Accuracy: 0.85

Random Forest Accuracy: 0.965

SVC Accuracy: 0.78

--- Detailed Report (Random Forest) ---
              precision    recall  f1-score   support

           0       1.00      0.64      0.78        14
           1       0.96      1.00      0.98       149
           2       0.97      0.95      0.96        37

    accuracy                           0.96       200
   macro avg       0.98      0.86      0.91       200
weighted avg       0.97      0.96      0.96       200



## Final Model Comparison

I tried three models: SVC, Logistic Regression, and Random Forest.

From the results:

- Logistic Regression performed well but is simple and assumes linear relationships  
- SVC gave decent performance but struggled with capturing complex patterns  
- Random Forest performed the best among all models  

I think Random Forest works better because it can handle non-linear relationships and feature interactions effectively.

Based on the comparison, Random Forest is the most suitable model for this problem.